# Analisi de les distribucions de vots en T vs en HDX

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


df_votes = pd.read_csv(r"../../data/all_surveys(votes).csv")

df_cmp = df_votes.copy()

# ============================================================
# CONFIG
# ============================================================

hdx_col = "<HDX-HDX_fixed+<HDX>>"
temp_col = "<T-T_fixed+<T>>"   # change if needed: "<T-T_fixed+<T>>"
comfort_col = "thermal_comfort"

# Convert numeric
df_cmp[hdx_col] = pd.to_numeric(df_cmp[hdx_col], errors="coerce")
df_cmp[temp_col] = pd.to_numeric(df_cmp[temp_col], errors="coerce")

# Convert Kelvin to Celsius if needed
if temp_col == "T_K" and df_cmp[temp_col].median() > 100:
    df_cmp["T_C"] = df_cmp[temp_col] - 273.15
    temp_used = "T_C"
else:
    temp_used = temp_col

# ============================================================
# 3-GROUP COMFORT DEFINITION
# ============================================================

comfort_to_group = {
    "Very comfortable": "1. Acceptable / neutral",
    "Comfortable": "1. Acceptable / neutral",
    "Slightly comfortable": "1. Acceptable / neutral",
    "Neutral": "1. Acceptable / neutral",

    "Slightly uncomfortable": "2. Discomfort",
    "Uncomfortable": "2. Discomfort",

    "Very uncomfortable": "3. Extreme discomfort",
}

group_order = [
    "1. Acceptable / neutral",
    "2. Discomfort",
    "3. Extreme discomfort",
]

comfort_order_7 = [
    "Very comfortable",
    "Comfortable",
    "Slightly comfortable",
    "Neutral",
    "Slightly uncomfortable",
    "Uncomfortable",
    "Very uncomfortable",
]

df_cmp["comfort_group"] = df_cmp[comfort_col].map(comfort_to_group)

df_cmp = df_cmp.dropna(
    subset=[hdx_col, temp_used, comfort_col, "comfort_group"]
).copy()

print("N:", len(df_cmp))
print("\nTemperature variable:", temp_used)
print("\nHDX summary:")
print(df_cmp[hdx_col].describe().round(3))
print("\nTemperature summary:")
print(df_cmp[temp_used].describe().round(3))
df_votes.columns

# parada tècnica per a calcular el pendent de l'ajust de m = P(U)- P(C)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from scipy import stats


# ============================================================
# CONFIG
# ============================================================

temp_abs_col = "<T>"
temp_corr_col = "<T-T_fixed+<T>>"
hdx_col = "<HDX-HDX_fixed+<HDX>>"

comfort_col = "thermal_comfort"

class_order = [
    "comfortable_side",
    "neutral",
    "uncomfortable_side",
]


# ============================================================
# PREPARE DATA
# ============================================================

df = df_votes.copy()
df["walk"] = df["space_code"].astype(str).str[:2]

for c in [temp_abs_col, temp_corr_col, hdx_col]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["T_rel_walk"] = (
    df[temp_abs_col]
    - df.groupby("walk")[temp_abs_col].transform("mean")
)


def comfort_side(cat):
    if cat in ["Very comfortable", "Comfortable", "Slightly comfortable"]:
        return "comfortable_side"
    elif cat == "Neutral":
        return "neutral"
    elif cat in ["Slightly uncomfortable", "Uncomfortable", "Very uncomfortable"]:
        return "uncomfortable_side"
    return np.nan


df["comfort_3"] = df[comfort_col].map(comfort_side)


# ============================================================
# HELPERS
# ============================================================

def wilson_ci(k, n, confidence=0.95):
    if n == 0:
        return np.nan, np.nan

    z = stats.norm.ppf(1 - (1 - confidence) / 2)
    phat = k / n

    denom = 1 + z**2 / n
    center = (phat + z**2 / (2 * n)) / denom
    half = (
        z
        * np.sqrt((phat * (1 - phat) / n) + (z**2 / (4 * n**2)))
        / denom
    )

    return center - half, center + half


def predict_proba_fixed_order(model, x_grid, class_order):
    probs = model.predict_proba(x_grid.reshape(-1, 1))

    prob_df = pd.DataFrame(probs, columns=model.classes_)

    for c in class_order:
        if c not in prob_df.columns:
            prob_df[c] = np.nan

    return prob_df[class_order].to_numpy()


def crossing_x(x, y):
    """
    Estimate x where y crosses 0.
    If no crossing exists, returns the x where |y| is smallest.
    """
    y = np.asarray(y, dtype=float)
    x = np.asarray(x, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) == 0:
        return np.nan

    sign_change = np.where(np.sign(y[:-1]) != np.sign(y[1:]))[0]

    if len(sign_change) == 0:
        return x[np.nanargmin(np.abs(y))]

    i = sign_change[0]

    x0, x1 = x[i], x[i + 1]
    y0, y1 = y[i], y[i + 1]

    if y1 == y0:
        return x0

    return x0 - y0 * (x1 - x0) / (y1 - y0)


def local_linear_slope(x, y, x0, half_window=None, min_points=8):
    """
    Approximate local slope dy/dx around x0.

    This fits a small straight line locally:
        y ≈ slope * x + intercept

    The result is more stable than taking a raw numerical derivative.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)

    x = x[ok]
    y = y[ok]

    if len(x) < 3 or not np.isfinite(x0):
        return np.nan

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if half_window is None:
        # Local window = 5% of total x-range on each side
        half_window = 0.05 * (np.nanmax(x) - np.nanmin(x))

    mask = (x >= x0 - half_window) & (x <= x0 + half_window)

    # Fallback: use nearest points if too few points in window
    if mask.sum() < min_points:
        nearest = np.argsort(np.abs(x - x0))[:min_points]
        mask = np.zeros_like(x, dtype=bool)
        mask[nearest] = True

    slope, intercept = np.polyfit(x[mask], y[mask], deg=1)

    return slope


def safe_nanpercentile(a, q):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]

    if len(a) == 0:
        return np.full(len(np.atleast_1d(q)), np.nan)

    return np.nanpercentile(a, q)


def safe_nanstd(a):
    a = np.asarray(a, dtype=float)
    a = a[np.isfinite(a)]

    if len(a) <= 1:
        return np.nan

    return np.nanstd(a, ddof=1)


# ============================================================
# MAIN FUNCTION WITH WALK-LEVEL BOOTSTRAP
# ============================================================

def fit_comfort_side_model_with_error(
    df_in,
    x_col,
    y_col="comfort_3",
    cluster_col="walk",
    n_boot=500,
    n_grid=500,
    n_bins=12,
    min_bin_n=10,
    local_slope_half_window=None,
    random_state=123,
):
    rng = np.random.default_rng(random_state)

    sub = df_in[[x_col, y_col, cluster_col]].dropna().copy()
    sub = sub[sub[y_col].isin(class_order)].copy()

    if sub.empty:
        print(f"No data for {x_col}")
        return None

    print(f"\nPredictor: {x_col}")
    print(f"N = {len(sub)}")
    print(sub[y_col].value_counts().reindex(class_order))

    # ----------------------------
    # Fit original model
    # ----------------------------

    X = sub[[x_col]].to_numpy()
    y = sub[y_col].to_numpy()

    model = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=3000,
    )

    model.fit(X, y)

    x_min, x_max = np.nanpercentile(sub[x_col], [1, 99])

    if x_min == x_max:
        print("x range is zero. Cannot fit a meaningful curve.")
        return None

    x_grid = np.linspace(x_min, x_max, n_grid)

    probs_hat = predict_proba_fixed_order(model, x_grid, class_order)

    prob_df = pd.DataFrame(probs_hat, columns=class_order)
    prob_df[x_col] = x_grid

    prob_df["m_order"] = (
        prob_df["uncomfortable_side"]
        - prob_df["comfortable_side"]
    )

    threshold_equal = crossing_x(
        x_grid,
        prob_df["uncomfortable_side"] - prob_df["comfortable_side"],
    )

    threshold_u_1_3 = crossing_x(
        x_grid,
        prob_df["uncomfortable_side"] - (1 / 3),
    )

    threshold_c_1_3 = crossing_x(
        x_grid,
        prob_df["comfortable_side"] - (1 / 3),
    )

    # ----------------------------
    # Original local slopes at P(U)=P(C)
    # ----------------------------

    slope_m_equal = local_linear_slope(
        x_grid,
        prob_df["m_order"].to_numpy(),
        threshold_equal,
        half_window=local_slope_half_window,
    )

    slope_u_equal = local_linear_slope(
        x_grid,
        prob_df["uncomfortable_side"].to_numpy(),
        threshold_equal,
        half_window=local_slope_half_window,
    )

    slope_c_equal = local_linear_slope(
        x_grid,
        prob_df["comfortable_side"].to_numpy(),
        threshold_equal,
        half_window=local_slope_half_window,
    )

    if np.isfinite(slope_m_equal) and slope_m_equal != 0:
        x_per_unit_m = 1 / slope_m_equal
    else:
        x_per_unit_m = np.nan

    # ----------------------------
    # Bootstrap
    # ----------------------------

    boot_probs = []
    boot_m = []

    boot_threshold_equal = []
    boot_threshold_u_1_3 = []
    boot_threshold_c_1_3 = []

    # Slopes at the P(U)=P(C) transition
    boot_slope_m_equal = []
    boot_slope_u_equal = []
    boot_slope_c_equal = []

    clusters = sub[cluster_col].dropna().unique()

    attempts = 0

    while len(boot_probs) < n_boot and attempts < n_boot * 10:
        attempts += 1

        sampled_clusters = rng.choice(
            clusters,
            size=len(clusters),
            replace=True,
        )

        boot_parts = []

        for cl in sampled_clusters:
            boot_parts.append(sub[sub[cluster_col] == cl])

        boot = pd.concat(boot_parts, axis=0)

        # Need all classes present
        if boot[y_col].nunique() < len(class_order):
            continue

        Xb = boot[[x_col]].to_numpy()
        yb = boot[y_col].to_numpy()

        try:
            mb = LogisticRegression(
                multi_class="multinomial",
                solver="lbfgs",
                max_iter=3000,
            )

            mb.fit(Xb, yb)

            pb = predict_proba_fixed_order(mb, x_grid, class_order)

            p_u_b = pb[:, class_order.index("uncomfortable_side")]
            p_c_b = pb[:, class_order.index("comfortable_side")]

            m_b = p_u_b - p_c_b

            boot_probs.append(pb)
            boot_m.append(m_b)

            # Threshold where P(U)=P(C)
            th_equal_b = crossing_x(x_grid, m_b)
            boot_threshold_equal.append(th_equal_b)

            # Local slopes at that threshold
            boot_slope_m_equal.append(
                local_linear_slope(
                    x_grid,
                    m_b,
                    th_equal_b,
                    half_window=local_slope_half_window,
                )
            )

            boot_slope_u_equal.append(
                local_linear_slope(
                    x_grid,
                    p_u_b,
                    th_equal_b,
                    half_window=local_slope_half_window,
                )
            )

            boot_slope_c_equal.append(
                local_linear_slope(
                    x_grid,
                    p_c_b,
                    th_equal_b,
                    half_window=local_slope_half_window,
                )
            )

            # Other thresholds
            boot_threshold_u_1_3.append(
                crossing_x(
                    x_grid,
                    p_u_b - (1 / 3),
                )
            )

            boot_threshold_c_1_3.append(
                crossing_x(
                    x_grid,
                    p_c_b - (1 / 3),
                )
            )

        except Exception:
            continue

    boot_probs = np.asarray(boot_probs)
    boot_m = np.asarray(boot_m)

    boot_threshold_equal = np.asarray(boot_threshold_equal)
    boot_threshold_u_1_3 = np.asarray(boot_threshold_u_1_3)
    boot_threshold_c_1_3 = np.asarray(boot_threshold_c_1_3)

    boot_slope_m_equal = np.asarray(boot_slope_m_equal)
    boot_slope_u_equal = np.asarray(boot_slope_u_equal)
    boot_slope_c_equal = np.asarray(boot_slope_c_equal)

    print(f"Bootstrap fits used: {len(boot_probs)} / {n_boot}")

    if len(boot_probs) == 0:
        print("No successful bootstrap fits. Returning original model only.")
        return {
            "predictor": x_col,
            "n": len(sub),
            "model": model,
            "prob_df": prob_df,
            "x_grid": x_grid,
            "threshold_PU_eq_PC": threshold_equal,
            "slope_m_at_PU_eq_PC": slope_m_equal,
            "slope_PU_at_PU_eq_PC": slope_u_equal,
            "slope_PC_at_PU_eq_PC": slope_c_equal,
        }

    # ----------------------------
    # Bootstrap CIs
    # ----------------------------

    # Probability CI
    prob_low = np.nanpercentile(boot_probs, 2.5, axis=0)
    prob_high = np.nanpercentile(boot_probs, 97.5, axis=0)

    # Order parameter CI
    m_low = np.nanpercentile(boot_m, 2.5, axis=0)
    m_high = np.nanpercentile(boot_m, 97.5, axis=0)

    # Threshold CIs
    th_equal_ci = safe_nanpercentile(boot_threshold_equal, [2.5, 50, 97.5])
    th_u_1_3_ci = safe_nanpercentile(boot_threshold_u_1_3, [2.5, 50, 97.5])
    th_c_1_3_ci = safe_nanpercentile(boot_threshold_c_1_3, [2.5, 50, 97.5])

    # Slope CIs
    slope_m_equal_ci = safe_nanpercentile(boot_slope_m_equal, [2.5, 50, 97.5])
    slope_u_equal_ci = safe_nanpercentile(boot_slope_u_equal, [2.5, 50, 97.5])
    slope_c_equal_ci = safe_nanpercentile(boot_slope_c_equal, [2.5, 50, 97.5])

    # Slope bootstrap standard errors
    slope_m_equal_se = safe_nanstd(boot_slope_m_equal)
    slope_u_equal_se = safe_nanstd(boot_slope_u_equal)
    slope_c_equal_se = safe_nanstd(boot_slope_c_equal)

    # Bootstrap inverse scale: x needed to change m by 1
    boot_x_per_unit_m = np.where(
        np.abs(boot_slope_m_equal) > 1e-12,
        1 / boot_slope_m_equal,
        np.nan,
    )

    x_per_unit_m_ci = safe_nanpercentile(boot_x_per_unit_m, [2.5, 50, 97.5])
    x_per_unit_m_se = safe_nanstd(boot_x_per_unit_m)

    # ----------------------------
    # Print results
    # ----------------------------

    print("\nThreshold estimates with bootstrap 95% CI:")
    print(
        f"P(U)=P(C): "
        f"{threshold_equal:.3f} "
        f"[{th_equal_ci[0]:.3f}, {th_equal_ci[2]:.3f}]"
    )

    print(
        f"P(U)=1/3: "
        f"{threshold_u_1_3:.3f} "
        f"[{th_u_1_3_ci[0]:.3f}, {th_u_1_3_ci[2]:.3f}]"
    )

    print(
        f"P(C)=1/3: "
        f"{threshold_c_1_3:.3f} "
        f"[{th_c_1_3_ci[0]:.3f}, {th_c_1_3_ci[2]:.3f}]"
    )

    print("\nApproximate local slopes at P(U)=P(C):")
    print(
        f"dm/dx = d[P(U)-P(C)]/dx: "
        f"{slope_m_equal:.5f} "
        f"± {slope_m_equal_se:.5f} bootstrap SE "
        f"[{slope_m_equal_ci[0]:.5f}, {slope_m_equal_ci[2]:.5f}]"
    )

    print(
        f"dP(U)/dx: "
        f"{slope_u_equal:.5f} "
        f"± {slope_u_equal_se:.5f} bootstrap SE "
        f"[{slope_u_equal_ci[0]:.5f}, {slope_u_equal_ci[2]:.5f}]"
    )

    print(
        f"dP(C)/dx: "
        f"{slope_c_equal:.5f} "
        f"± {slope_c_equal_se:.5f} bootstrap SE "
        f"[{slope_c_equal_ci[0]:.5f}, {slope_c_equal_ci[2]:.5f}]"
    )

    print("\nApproximate inverse transition scale:")
    print(
        f"1 / (dm/dx): "
        f"{x_per_unit_m:.5f} "
        f"± {x_per_unit_m_se:.5f} bootstrap SE "
        f"[{x_per_unit_m_ci[0]:.5f}, {x_per_unit_m_ci[2]:.5f}]"
    )

    # ----------------------------
    # Empirical binned probabilities with Wilson CI
    # ----------------------------

    rows = []

    try:
        sub["x_bin"] = pd.qcut(
            sub[x_col],
            q=n_bins,
            duplicates="drop",
        )

        for b, g in sub.groupby("x_bin", observed=False):
            n = len(g)

            if n < min_bin_n:
                continue

            row = {
                "x_mean": g[x_col].mean(),
                "x_median": g[x_col].median(),
                "n": n,
            }

            for cls in class_order:
                k = (g[y_col] == cls).sum()
                p = k / n
                lo, hi = wilson_ci(k, n)

                row[f"{cls}_p"] = p
                row[f"{cls}_lo"] = lo
                row[f"{cls}_hi"] = hi
                row[f"{cls}_k"] = k

            rows.append(row)

        bin_df = pd.DataFrame(rows)

    except Exception:
        bin_df = pd.DataFrame()

    # ----------------------------
    # Plot 1: probabilities with bootstrap bands
    # ----------------------------

    plt.figure(figsize=(9, 5))

    for i, cls in enumerate(class_order):
        plt.fill_between(
            x_grid,
            prob_low[:, i],
            prob_high[:, i],
            alpha=0.18,
        )

        plt.plot(
            x_grid,
            probs_hat[:, i],
            linewidth=2,
            label=f"P({cls})",
        )

        if not bin_df.empty:
            y_bin = bin_df[f"{cls}_p"].values
            yerr = [
                y_bin - bin_df[f"{cls}_lo"].values,
                bin_df[f"{cls}_hi"].values - y_bin,
            ]

            plt.errorbar(
                bin_df["x_mean"],
                y_bin,
                yerr=yerr,
                fmt="o",
                capsize=3,
                alpha=0.65,
            )

    plt.axhline(
        1 / 3,
        linestyle="--",
        alpha=0.5,
        label="1/3",
    )

    plt.axvline(
        threshold_equal,
        linestyle="--",
        alpha=0.8,
        label=f"P(U)=P(C): {threshold_equal:.2f}",
    )

    plt.axvspan(
        th_equal_ci[0],
        th_equal_ci[2],
        alpha=0.12,
        label="95% CI threshold",
    )

    plt.axvline(
        threshold_u_1_3,
        linestyle=":",
        alpha=0.8,
        label=f"P(U)=1/3: {threshold_u_1_3:.2f}",
    )

    plt.ylim(-0.02, 1.02)
    plt.xlabel(x_col)
    plt.ylabel("Predicted probability")
    plt.title(
        f"Comfort-side probabilities vs {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Plot 2: order parameter with bootstrap band and local slope
    # ----------------------------

    plt.figure(figsize=(9, 4.5))

    plt.fill_between(
        x_grid,
        m_low,
        m_high,
        alpha=0.22,
        label="95% bootstrap CI",
    )

    plt.plot(
        x_grid,
        prob_df["m_order"],
        linewidth=2,
        label="m = P(U) - P(C)",
    )

    plt.axhline(0, linestyle="--", alpha=0.7)

    plt.axvline(
        threshold_equal,
        linestyle="--",
        alpha=0.8,
        label=f"m=0 at {threshold_equal:.2f}",
    )

    plt.axvspan(
        th_equal_ci[0],
        th_equal_ci[2],
        alpha=0.12,
        label="95% CI threshold",
    )

    # Local tangent approximation around the transition
    tangent_half_width = 0.08 * (x_grid.max() - x_grid.min())

    x_tangent = np.linspace(
        threshold_equal - tangent_half_width,
        threshold_equal + tangent_half_width,
        100,
    )

    m_tangent = slope_m_equal * (x_tangent - threshold_equal)

    plt.plot(
        x_tangent,
        m_tangent,
        linestyle="-.",
        linewidth=2,
        alpha=0.9,
        label=f"Local slope ≈ {slope_m_equal:.3f}",
    )

    plt.xlabel(x_col)
    plt.ylabel("Order parameter m")
    plt.title(
        f"P(discomfort) - P(comfort) vs {x_col}\n"
        "Bands = walk-level bootstrap 95% CI"
    )
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Plot 3: threshold bootstrap distribution
    # ----------------------------

    plt.figure(figsize=(8, 4.5))

    plt.hist(
        boot_threshold_equal,
        bins=30,
        alpha=0.75,
    )

    plt.axvline(threshold_equal, linestyle="--", label="Original estimate")
    plt.axvline(th_equal_ci[0], linestyle=":", label="2.5%")
    plt.axvline(th_equal_ci[2], linestyle=":", label="97.5%")

    plt.xlabel(x_col)
    plt.ylabel("Bootstrap count")
    plt.title("Bootstrap uncertainty of P(U)=P(C) threshold")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Plot 4: slope bootstrap distribution
    # ----------------------------

    plt.figure(figsize=(8, 4.5))

    plt.hist(
        boot_slope_m_equal,
        bins=30,
        alpha=0.75,
    )

    plt.axvline(
        slope_m_equal,
        linestyle="--",
        label="Original slope",
    )

    plt.axvline(
        slope_m_equal_ci[0],
        linestyle=":",
        label="2.5%",
    )

    plt.axvline(
        slope_m_equal_ci[2],
        linestyle=":",
        label="97.5%",
    )

    plt.xlabel(f"dm/d{x_col}")
    plt.ylabel("Bootstrap count")
    plt.title("Bootstrap uncertainty of local transition slope")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # Return results
    # ----------------------------

    result = {
        "predictor": x_col,
        "n": len(sub),

        "model": model,
        "prob_df": prob_df,
        "bin_df": bin_df,
        "x_grid": x_grid,

        "boot_probs": boot_probs,
        "boot_m": boot_m,

        "threshold_PU_eq_PC": threshold_equal,
        "threshold_PU_eq_PC_CI": th_equal_ci,

        "threshold_PU_eq_1_3": threshold_u_1_3,
        "threshold_PU_eq_1_3_CI": th_u_1_3_ci,

        "threshold_PC_eq_1_3": threshold_c_1_3,
        "threshold_PC_eq_1_3_CI": th_c_1_3_ci,

        "slope_m_at_PU_eq_PC": slope_m_equal,
        "slope_m_at_PU_eq_PC_SE": slope_m_equal_se,
        "slope_m_at_PU_eq_PC_CI": slope_m_equal_ci,

        "slope_PU_at_PU_eq_PC": slope_u_equal,
        "slope_PU_at_PU_eq_PC_SE": slope_u_equal_se,
        "slope_PU_at_PU_eq_PC_CI": slope_u_equal_ci,

        "slope_PC_at_PU_eq_PC": slope_c_equal,
        "slope_PC_at_PU_eq_PC_SE": slope_c_equal_se,
        "slope_PC_at_PU_eq_PC_CI": slope_c_equal_ci,

        "x_per_unit_m": x_per_unit_m,
        "x_per_unit_m_SE": x_per_unit_m_se,
        "x_per_unit_m_CI": x_per_unit_m_ci,

        "boot_slope_m_at_PU_eq_PC": boot_slope_m_equal,
        "boot_slope_PU_at_PU_eq_PC": boot_slope_u_equal,
        "boot_slope_PC_at_PU_eq_PC": boot_slope_c_equal,
    }

    return result

In [ ]:
res_temp_corr = fit_comfort_side_model_with_error(
    df,
    x_col=temp_corr_col,
    n_boot=500,
    n_grid=500,
    n_bins=12,
    min_bin_n=10,
    random_state=123,
)

In [ ]:
res_temp_corr["slope_m_at_PU_eq_PC"]
res_temp_corr["slope_m_at_PU_eq_PC_SE"]
res_temp_corr["slope_m_at_PU_eq_PC_CI"]
res_temp_corr["slope_PU_at_PU_eq_PC"]

## Pero aquestes son probabilitats de votar uncomfortable side ii estas cosas, però no son la probabilitat de augmentar el teu vot (que és el que teniem abans). Per comparar necessitarem la probabilitat de transició (d'empitjorament)

## serà agafar en 7 categories, mirar totes les transiions poistives i miarr com canvien en T

## també en 3 categories : PERÒ SEGUIM TENINT PROBABILIITATS -> però tindrà més sentit que 1/2 vegades tinguem un canvi d'1TCV per cada x graus, que és el que podrem obtenir

# enough stop, ara seguim amb distribucions de HDX vs T

In [ ]:
def binned_vote_distribution(
    df,
    x_col,
    label_col,
    label_order,
    n_bins=15,
    binning="equal_width",  # "equal_width" or "quantile"
    min_bin_n=10,
):
    d = df[[x_col, label_col]].dropna().copy()
    d = d[d[label_col].isin(label_order)].copy()

    if binning == "equal_width":
        d["x_bin"] = pd.cut(
            d[x_col],
            bins=n_bins,
            include_lowest=True,
        )
    elif binning == "quantile":
        d["x_bin"] = pd.qcut(
            d[x_col],
            q=n_bins,
            duplicates="drop",
        )
    else:
        raise ValueError("binning must be 'equal_width' or 'quantile'")

    counts = pd.crosstab(
        d[label_col],
        d["x_bin"],
    ).reindex(label_order)

    bin_counts = counts.sum(axis=0)

    probs = counts.div(bin_counts, axis=1)

    # bin summary
    rows = []
    for b, g in d.groupby("x_bin", observed=False):
        rows.append({
            "bin": b,
            "x_mean": g[x_col].mean(),
            "x_median": g[x_col].median(),
            "x_min": g[x_col].min(),
            "x_max": g[x_col].max(),
            "n": len(g),
        })

    bin_summary = pd.DataFrame(rows).sort_values("x_mean")

    # Mask bins with low n
    probs_masked = probs.copy()
    probs_masked.loc[:, bin_counts < min_bin_n] = np.nan

    return counts, probs, probs_masked, bin_counts, bin_summary

In [ ]:
def plot_coverage_and_heatmap(
    df,
    x_col,
    label_col,
    label_order,
    title,
    n_bins=15,
    binning="equal_width",
    min_bin_n=10,
    cmap_name="viridis",
):
    counts, probs, probs_masked, bin_counts, bin_summary = binned_vote_distribution(
        df=df,
        x_col=x_col,
        label_col=label_col,
        label_order=label_order,
        n_bins=n_bins,
        binning=binning,
        min_bin_n=min_bin_n,
    )

    x_centers = bin_summary["x_mean"].values

    fig, axes = plt.subplots(
        2, 1,
        figsize=(11, 6.5),
        gridspec_kw={"height_ratios": [1, 4]},
        sharex=True,
    )

    ax_top, ax = axes

    # ----------------------------
    # Top: vote coverage
    # ----------------------------
    ax_top.bar(
        x_centers,
        bin_summary["n"].values,
        width=np.diff(x_centers).mean() * 0.8 if len(x_centers) > 1 else 0.5,
        alpha=0.75,
    )

    ax_top.set_ylabel("Votes\nper bin")
    ax_top.set_title(title)
    ax_top.grid(True, axis="y", alpha=0.3)

    # ----------------------------
    # Bottom: P(label | bin)
    # ----------------------------
    cmap = plt.cm.get_cmap(cmap_name).copy()
    cmap.set_bad(color="white")

    im = ax.imshow(
        probs_masked.values,
        aspect="auto",
        interpolation="nearest",
        cmap=cmap,
        extent=[
            x_centers.min(),
            x_centers.max(),
            len(label_order),
            0,
        ],
        vmin=0,
        vmax=np.nanmax(probs_masked.values),
    )

    ax.set_yticks(np.arange(len(label_order)) + 0.5)
    ax.set_yticklabels(label_order)

    ax.set_xlabel(x_col)
    ax.set_ylabel("Comfort state")
    ax.set_title(
        f"P(state | {x_col} bin), bins with n < {min_bin_n} masked"
    )

    fig.colorbar(im, ax=ax, label="Probability within bin")

    plt.tight_layout()
    plt.show()

    return counts, probs, bin_counts, bin_summary

In [ ]:
# ============================================================
# HDX — 3 groups
# ============================================================

hdx_counts_3, hdx_probs_3, hdx_bin_counts_3, hdx_summary_3 = plot_coverage_and_heatmap(
    df=df_cmp,
    x_col=hdx_col,
    label_col="comfort_group",
    label_order=group_order,
    title="HDX coverage and 3-group comfort distribution",
    n_bins=15,
    binning="equal_width",
    min_bin_n=10,
)

# ============================================================
# Temperature — 3 groups
# ============================================================

t_counts_3, t_probs_3, t_bin_counts_3, t_summary_3 = plot_coverage_and_heatmap(
    df=df_cmp,
    x_col=temp_used,
    label_col="comfort_group",
    label_order=group_order,
    title="Temperature coverage and 3-group comfort distribution",
    n_bins=15,
    binning="equal_width",
    min_bin_n=10,
)

In [ ]:
# ============================================================
# HDX — 7 categories
# ============================================================

hdx_counts_7, hdx_probs_7, hdx_bin_counts_7, hdx_summary_7 = plot_coverage_and_heatmap(
    df=df_cmp,
    x_col=hdx_col,
    label_col=comfort_col,
    label_order=comfort_order_7,
    title="HDX coverage and 7-category comfort distribution",
    n_bins=15,
    binning="equal_width",
    min_bin_n=10,
)

# ============================================================
# Temperature — 7 categories
# ============================================================

t_counts_7, t_probs_7, t_bin_counts_7, t_summary_7 = plot_coverage_and_heatmap(
    df=df_cmp,
    x_col=temp_used,
    label_col=comfort_col,
    label_order=comfort_order_7,
    title="Temperature coverage and 7-category comfort distribution",
    n_bins=15,
    binning="equal_width",
    min_bin_n=10,
)

In [ ]:
coverage_summary = pd.DataFrame({
    "variable": ["HDX", "Temperature"],
    "n": [
        df_cmp[hdx_col].notna().sum(),
        df_cmp[temp_used].notna().sum(),
    ],
    "min": [
        df_cmp[hdx_col].min(),
        df_cmp[temp_used].min(),
    ],
    "q25": [
        df_cmp[hdx_col].quantile(0.25),
        df_cmp[temp_used].quantile(0.25),
    ],
    "median": [
        df_cmp[hdx_col].median(),
        df_cmp[temp_used].median(),
    ],
    "q75": [
        df_cmp[hdx_col].quantile(0.75),
        df_cmp[temp_used].quantile(0.75),
    ],
    "max": [
        df_cmp[hdx_col].max(),
        df_cmp[temp_used].max(),
    ],
    "IQR": [
        df_cmp[hdx_col].quantile(0.75) - df_cmp[hdx_col].quantile(0.25),
        df_cmp[temp_used].quantile(0.75) - df_cmp[temp_used].quantile(0.25),
    ],
})

coverage_summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].hist(df_cmp[hdx_col].dropna(), bins=20, alpha=0.75)
axes[0].set_xlabel("HDX")
axes[0].set_ylabel("Votes")
axes[0].set_title("Vote distribution across HDX")
axes[0].grid(True, alpha=0.3)

axes[1].hist(df_cmp[temp_used].dropna(), bins=20, alpha=0.75)
axes[1].set_xlabel("Temperature")
axes[1].set_ylabel("Votes")
axes[1].set_title("Vote distribution across temperature")
axes[1].grid(True, alpha=0.3)

plt.show()